In [1]:
# ==========================================================
# Banking Stress Testing Framework
# Module 02 - Data Preparation
# ==========================================================

# =============================================================================
# PROJECT       : Banking Stress Testing Framework
# VERSION       : Python V2 Migration
# MODULE        : 02 - Data Preparation
# NOTEBOOK      : 02_Data_Preparation.ipynb
# AUTHOR        : Umesh Zure
#
# PURPOSE:
# Prepare Module 01 output for downstream risk segmentation and stress testing.
#
# INPUT:
# output/portfolio_raw.csv
#
# OUTPUTS:
# output/prepared_portfolio.csv
# validation/02_data_preparation_summary.csv
# validation/02_data_preparation_validation.csv
# logs/02_Data_Preparation.log
#
# BUSINESS RULES:
# 1. Standardize categorical variables.
# 2. Impute missing Credit_Score using Customer_Segment median.
# 3. Use portfolio median as fallback for Credit_Score.
# 4. Impute missing Current_DPD with zero.
# 5. Apply valid Credit_Score and Current_DPD boundaries.
# 6. Preserve missing Collateral_Value values.
# 7. Exclude segmentation, which will be performed in Module 03.
# =============================================================================

print("=" * 80)
print("BANKING STRESS TESTING FRAMEWORK")
print("PYTHON V2 - MODULE 02: DATA PREPARATION")
print("=" * 80)

BANKING STRESS TESTING FRAMEWORK
PYTHON V2 - MODULE 02: DATA PREPARATION


In [2]:
# =============================================================================
# BLOCK 1: ENVIRONMENT AND PATH CONFIGURATION
# =============================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime
import logging
import sys

import numpy as np
import pandas as pd

# Mount Google Drive.
drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Project folders and files
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Banking_Stress_Testing_V2_Python"
)

INPUT_FILE = (
    PROJECT_ROOT
    / "output"
    / "portfolio_raw.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "output"
    / "prepared_portfolio.csv"
)

VALIDATION_DIR = (
    PROJECT_ROOT
    / "validation"
)

LOG_DIR = (
    PROJECT_ROOT
    / "logs"
)

SUMMARY_FILE = (
    VALIDATION_DIR
    / "02_data_preparation_summary.csv"
)

VALIDATION_FILE = (
    VALIDATION_DIR
    / "02_data_preparation_validation.csv"
)

LOG_FILE = (
    LOG_DIR
    / "02_Data_Preparation.log"
)

# Create output directories where necessary.
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------

logger = logging.getLogger(
    "02_Data_Preparation"
)

logger.setLevel(
    logging.INFO
)

# Prevent duplicate logging when rerunning this cell.
if logger.hasHandlers():
    logger.handlers.clear()

log_format = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

file_handler = logging.FileHandler(
    LOG_FILE,
    mode="w",
    encoding="utf-8"
)

file_handler.setFormatter(
    log_format
)

console_handler = logging.StreamHandler(
    sys.stdout
)

console_handler.setFormatter(
    log_format
)

logger.addHandler(
    file_handler
)

logger.addHandler(
    console_handler
)

# ---------------------------------------------------------------------------
# Path validation
# ---------------------------------------------------------------------------

print("=" * 80)
print("ENVIRONMENT AND PATH CONFIGURATION")
print("=" * 80)

print(
    "Project Root Exists:",
    PROJECT_ROOT.exists()
)

print(
    "Module 01 Output Exists:",
    INPUT_FILE.exists()
)

print(
    "Input File:",
    INPUT_FILE
)

print(
    "Prepared Output File:",
    OUTPUT_FILE
)

print(
    "Validation Directory:",
    VALIDATION_DIR
)

print(
    "Log File:",
    LOG_FILE
)

assert PROJECT_ROOT.exists(), (
    f"Project folder was not found: {PROJECT_ROOT}"
)

assert INPUT_FILE.exists(), (
    f"Module 01 output was not found: {INPUT_FILE}"
)

logger.info(
    "Environment and path configuration completed successfully."
)



Mounted at /content/drive
ENVIRONMENT AND PATH CONFIGURATION
Project Root Exists: True
Module 01 Output Exists: True
Input File: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_raw.csv
Prepared Output File: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/prepared_portfolio.csv
Validation Directory: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation
Log File: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/02_Data_Preparation.log
2026-10-13 03:32:57,090 | INFO | Environment and path configuration completed successfully.


INFO:02_Data_Preparation:Environment and path configuration completed successfully.


In [3]:
# =============================================================================
# BLOCK 2: LOAD AND VALIDATE MODULE 01 OUTPUT
# =============================================================================

EXPECTED_COLUMNS = [
    "Customer_ID",
    "Loan_ID",
    "Customer_Segment",
    "Product_Type",
    "Outstanding_Balance",
    "Credit_Score",
    "Previous_Credit_Score",
    "Current_DPD",
    "Previous_DPD",
    "Interest_Rate",
    "Loan_Term_Months",
    "Remaining_Term_Months",
    "Collateral_Value",
    "Region",
    "Reporting_Date",
]

logger.info(
    "Starting Module 02 data preparation."
)

try:
    portfolio_raw = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

except FileNotFoundError as error:
    logger.exception(
        "Module 01 output was not found."
    )
    raise error

except pd.errors.EmptyDataError as error:
    logger.exception(
        "Module 01 output is empty."
    )
    raise error

except pd.errors.ParserError as error:
    logger.exception(
        "Module 01 output could not be parsed."
    )
    raise error

# Create an independent preparation dataset.
prepared_portfolio = portfolio_raw.copy(
    deep=True
)

input_record_count = len(
    prepared_portfolio
)

input_column_count = len(
    prepared_portfolio.columns
)

missing_columns = sorted(
    set(EXPECTED_COLUMNS)
    - set(prepared_portfolio.columns)
)

unexpected_columns = sorted(
    set(prepared_portfolio.columns)
    - set(EXPECTED_COLUMNS)
)

if missing_columns:
    logger.error(
        "Required columns are missing: %s",
        missing_columns
    )

    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("=" * 80)
print("MODULE 01 OUTPUT LOADED SUCCESSFULLY")
print("=" * 80)

print(
    "Input Records:",
    input_record_count
)

print(
    "Input Columns:",
    input_column_count
)

print(
    "Missing Required Columns:",
    missing_columns
)

print(
    "Unexpected Columns:",
    unexpected_columns
)

print(
    "Schema Status: PASS"
)

display(
    prepared_portfolio.head()
)

logger.info(
    "Module 01 output loaded: %s rows and %s columns.",
    input_record_count,
    input_column_count
)


2026-10-13 03:37:38,692 | INFO | Starting Module 02 data preparation.


INFO:02_Data_Preparation:Starting Module 02 data preparation.


MODULE 01 OUTPUT LOADED SUCCESSFULLY
Input Records: 10000
Input Columns: 15
Missing Required Columns: []
Unexpected Columns: []
Schema Status: PASS


,Customer_ID,Loan_ID,Customer_Segment,Product_Type,Outstanding_Balance,Credit_Score,Previous_Credit_Score,Current_DPD,Previous_DPD,Interest_Rate,Loan_Term_Months,Remaining_Term_Months,Collateral_Value,Region,Reporting_Date
0,1000001,50000001,Retail,Personal Loan,1733362.19,756.0,733,0.0,15,19.40,77,21,NaN,West,2026-09-30
1,1000002,50000002,Corporate,Corporate Loan,62244073.93,817.0,805,0.0,30,10.82,22,6,NaN,East,2026-09-30
2,1000003,50000003,Retail,Credit Card,303259.15,630.0,610,0.0,0,25.70,22,12,NaN,East,2026-09-30
3,1000004,50000004,SME,Auto Loan,993246.23,724.0,688,0.0,0,13.14,65,9,918420.09,West,2026-09-30
4,1000005,50000005,Retail,Personal Loan,1654985.49,784.0,761,0.0,0,16.05,25,21,NaN,South,2026-09-30


2026-10-13 03:37:39,146 | INFO | Module 01 output loaded: 10000 rows and 15 columns.


INFO:02_Data_Preparation:Module 01 output loaded: 10000 rows and 15 columns.


In [4]:
# =============================================================================
# BLOCK 3: CREDIT SCORE HANDLING
# =============================================================================

# Ensure Credit_Score is numeric.
prepared_portfolio[
    "Credit_Score"
] = pd.to_numeric(
    prepared_portfolio[
        "Credit_Score"
    ],
    errors="coerce"
)

# Preserve the original source value.
prepared_portfolio[
    "Original_Credit_Score"
] = prepared_portfolio[
    "Credit_Score"
]

# Identify source records requiring imputation.
prepared_portfolio[
    "Credit_Score_Imputed_Flag"
] = np.where(
    prepared_portfolio[
        "Credit_Score"
    ].isna(),
    1,
    0
)

missing_credit_score_before = int(
    prepared_portfolio[
        "Credit_Score"
    ]
    .isna()
    .sum()
)

# Calculate median Credit Score by Customer Segment.
segment_credit_score_median = (
    prepared_portfolio
    .groupby(
        "Customer_Segment",
        dropna=False
    )["Credit_Score"]
    .median()
)

# Calculate portfolio median as fallback.
portfolio_credit_score_median = float(
    prepared_portfolio[
        "Credit_Score"
    ]
    .median()
)

# Map each row to the applicable segment median.
segment_median_by_row = (
    prepared_portfolio[
        "Customer_Segment"
    ]
    .map(
        segment_credit_score_median
    )
)

# First imputation level: Customer Segment median.
prepared_portfolio[
    "Credit_Score"
] = prepared_portfolio[
    "Credit_Score"
].fillna(
    segment_median_by_row
)

# Second imputation level: portfolio median fallback.
prepared_portfolio[
    "Credit_Score"
] = prepared_portfolio[
    "Credit_Score"
].fillna(
    portfolio_credit_score_median
)

# Count boundary exceptions before corrections.
credit_score_below_300 = int(
    (
        prepared_portfolio[
            "Credit_Score"
        ] < 300
    )
    .sum()
)

credit_score_above_900 = int(
    (
        prepared_portfolio[
            "Credit_Score"
        ] > 900
    )
    .sum()
)

# Enforce valid Credit Score range.
prepared_portfolio[
    "Credit_Score"
] = prepared_portfolio[
    "Credit_Score"
].clip(
    lower=300,
    upper=900
)

credit_scores_imputed = int(
    prepared_portfolio[
        "Credit_Score_Imputed_Flag"
    ]
    .sum()
)

missing_credit_score_after = int(
    prepared_portfolio[
        "Credit_Score"
    ]
    .isna()
    .sum()
)

logger.info(
    "Credit Score preparation completed. "
    "Imputed records: %s.",
    credit_scores_imputed
)

print("=" * 80)
print("CREDIT SCORE PREPARATION COMPLETED")
print("=" * 80)

print(
    "Missing Credit Scores Before:",
    missing_credit_score_before
)

print(
    "Credit Scores Imputed:",
    credit_scores_imputed
)

print(
    "Scores Corrected Below 300:",
    credit_score_below_300
)

print(
    "Scores Corrected Above 900:",
    credit_score_above_900
)

print(
    "Missing Credit Scores After:",
    missing_credit_score_after
)

print(
    "Portfolio Credit Score Median:",
    portfolio_credit_score_median
)

print("\nCustomer Segment Credit Score Medians:")

display(
    segment_credit_score_median
    .rename(
        "Segment_Median_Credit_Score"
    )
    .reset_index()
)

print("\nSample Imputed Credit Score Records:")

display(
    prepared_portfolio.loc[
        prepared_portfolio[
            "Credit_Score_Imputed_Flag"
        ] == 1,
        [
            "Customer_ID",
            "Loan_ID",
            "Customer_Segment",
            "Original_Credit_Score",
            "Credit_Score",
            "Credit_Score_Imputed_Flag",
        ]
    ]
    .head(10)
)


2026-10-13 03:40:18,047 | INFO | Credit Score preparation completed. Imputed records: 100.


INFO:02_Data_Preparation:Credit Score preparation completed. Imputed records: 100.


CREDIT SCORE PREPARATION COMPLETED
Missing Credit Scores Before: 100
Credit Scores Imputed: 100
Scores Corrected Below 300: 0
Scores Corrected Above 900: 0
Missing Credit Scores After: 0
Portfolio Credit Score Median: 700.0

Customer Segment Credit Score Medians:


,Customer_Segment,Segment_Median_Credit_Score
0,Corporate,698.0
1,Retail,701.0
2,SME,698.0



Sample Imputed Credit Score Records:


,Customer_ID,Loan_ID,Customer_Segment,Original_Credit_Score,Credit_Score,Credit_Score_Imputed_Flag
258,1000259,50000259,Retail,NaN,701.0,1
383,1000384,50000384,Corporate,NaN,698.0,1
412,1000413,50000413,SME,NaN,698.0,1
509,1000510,50000510,Retail,NaN,701.0,1
709,1000710,50000710,Retail,NaN,701.0,1
881,1000882,50000882,Retail,NaN,701.0,1
956,1000957,50000957,Retail,NaN,701.0,1
1061,1001062,50001062,Retail,NaN,701.0,1
1084,1001085,50001085,SME,NaN,698.0,1
1130,1001131,50001131,Retail,NaN,701.0,1


In [5]:
# =============================================================================
# BLOCK 4: CURRENT DPD HANDLING
# =============================================================================

# Ensure Current_DPD is numeric.
prepared_portfolio[
    "Current_DPD"
] = pd.to_numeric(
    prepared_portfolio[
        "Current_DPD"
    ],
    errors="coerce"
)

# Preserve the original source value.
prepared_portfolio[
    "Original_Current_DPD"
] = prepared_portfolio[
    "Current_DPD"
]

# Identify records requiring imputation.
prepared_portfolio[
    "Current_DPD_Imputed_Flag"
] = np.where(
    prepared_portfolio[
        "Current_DPD"
    ].isna(),
    1,
    0
)

missing_current_dpd_before = int(
    prepared_portfolio[
        "Current_DPD"
    ]
    .isna()
    .sum()
)

negative_current_dpd_before = int(
    (
        prepared_portfolio[
            "Current_DPD"
        ] < 0
    )
    .sum()
)

current_dpd_above_999 = int(
    (
        prepared_portfolio[
            "Current_DPD"
        ] > 999
    )
    .sum()
)

# Missing Current_DPD is treated as zero.
prepared_portfolio[
    "Current_DPD"
] = prepared_portfolio[
    "Current_DPD"
].fillna(0)

# Apply controlled lower and upper boundaries.
prepared_portfolio[
    "Current_DPD"
] = prepared_portfolio[
    "Current_DPD"
].clip(
    lower=0,
    upper=999
)

current_dpd_imputed = int(
    prepared_portfolio[
        "Current_DPD_Imputed_Flag"
    ]
    .sum()
)

missing_current_dpd_after = int(
    prepared_portfolio[
        "Current_DPD"
    ]
    .isna()
    .sum()
)

logger.info(
    "Current DPD preparation completed. "
    "Imputed records: %s.",
    current_dpd_imputed
)

print("=" * 80)
print("CURRENT DPD PREPARATION COMPLETED")
print("=" * 80)

print(
    "Missing Current DPD Before:",
    missing_current_dpd_before
)

print(
    "Current DPD Values Imputed:",
    current_dpd_imputed
)

print(
    "Negative DPD Values Corrected:",
    negative_current_dpd_before
)

print(
    "DPD Values Corrected Above 999:",
    current_dpd_above_999
)

print(
    "Missing Current DPD After:",
    missing_current_dpd_after
)

print("\nSample Imputed Current DPD Records:")

display(
    prepared_portfolio.loc[
        prepared_portfolio[
            "Current_DPD_Imputed_Flag"
        ] == 1,
        [
            "Customer_ID",
            "Loan_ID",
            "Original_Current_DPD",
            "Current_DPD",
            "Current_DPD_Imputed_Flag",
        ]
    ]
    .head(10)
)

2026-10-13 03:42:03,275 | INFO | Current DPD preparation completed. Imputed records: 75.


INFO:02_Data_Preparation:Current DPD preparation completed. Imputed records: 75.


CURRENT DPD PREPARATION COMPLETED
Missing Current DPD Before: 75
Current DPD Values Imputed: 75
Negative DPD Values Corrected: 0
DPD Values Corrected Above 999: 0
Missing Current DPD After: 0

Sample Imputed Current DPD Records:


,Customer_ID,Loan_ID,Original_Current_DPD,Current_DPD,Current_DPD_Imputed_Flag
58,1000059,50000059,NaN,0.0,1
188,1000189,50000189,NaN,0.0,1
200,1000201,50000201,NaN,0.0,1
524,1000525,50000525,NaN,0.0,1
701,1000702,50000702,NaN,0.0,1
733,1000734,50000734,NaN,0.0,1
756,1000757,50000757,NaN,0.0,1
812,1000813,50000813,NaN,0.0,1
866,1000867,50000867,NaN,0.0,1
1191,1001192,50001192,NaN,0.0,1


In [6]:
# =============================================================================
# BLOCK 5: STANDARDIZATION AND BASIC BUSINESS VALIDATION
# =============================================================================

categorical_columns = [
    "Customer_Segment",
    "Product_Type",
    "Region",
]

# Trim leading and trailing spaces and reduce repeated spaces.
for column in categorical_columns:
    prepared_portfolio[column] = (
        prepared_portfolio[column]
        .astype("string")
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

# Controlled canonical values.
segment_map = {
    "RETAIL": "Retail",
    "SME": "SME",
    "CORPORATE": "Corporate",
}

product_map = {
    "MORTGAGE": "Mortgage",
    "AUTO LOAN": "Auto Loan",
    "PERSONAL LOAN": "Personal Loan",
    "CREDIT CARD": "Credit Card",
    "SME LOAN": "SME Loan",
    "CORPORATE LOAN": "Corporate Loan",
}

region_map = {
    "NORTH": "North",
    "SOUTH": "South",
    "EAST": "East",
    "WEST": "West",
    "CENTRAL": "Central",
}

prepared_portfolio[
    "Customer_Segment"
] = (
    prepared_portfolio[
        "Customer_Segment"
    ]
    .str.upper()
    .map(segment_map)
)

prepared_portfolio[
    "Product_Type"
] = (
    prepared_portfolio[
        "Product_Type"
    ]
    .str.upper()
    .map(product_map)
)

prepared_portfolio[
    "Region"
] = (
    prepared_portfolio[
        "Region"
    ]
    .str.upper()
    .map(region_map)
)

# Convert Reporting_Date to datetime.
prepared_portfolio[
    "Reporting_Date"
] = pd.to_datetime(
    prepared_portfolio[
        "Reporting_Date"
    ],
    errors="coerce"
)

# Ensure supporting fields are numeric.
numeric_columns = [
    "Outstanding_Balance",
    "Previous_Credit_Score",
    "Previous_DPD",
    "Interest_Rate",
    "Loan_Term_Months",
    "Remaining_Term_Months",
    "Collateral_Value",
]

for column in numeric_columns:
    prepared_portfolio[column] = pd.to_numeric(
        prepared_portfolio[column],
        errors="coerce"
    )

invalid_segments = int(
    prepared_portfolio[
        "Customer_Segment"
    ]
    .isna()
    .sum()
)

invalid_products = int(
    prepared_portfolio[
        "Product_Type"
    ]
    .isna()
    .sum()
)

invalid_regions = int(
    prepared_portfolio[
        "Region"
    ]
    .isna()
    .sum()
)

invalid_reporting_dates = int(
    prepared_portfolio[
        "Reporting_Date"
    ]
    .isna()
    .sum()
)

invalid_balances = int(
    (
        prepared_portfolio[
            "Outstanding_Balance"
        ] <= 0
    )
    .sum()
)

invalid_remaining_terms = int(
    (
        prepared_portfolio[
            "Remaining_Term_Months"
        ] < 0
    )
    .sum()
)

remaining_above_total_term = int(
    (
        prepared_portfolio[
            "Remaining_Term_Months"
        ]
        >
        prepared_portfolio[
            "Loan_Term_Months"
        ]
    )
    .sum()
)

secured_product_mask = (
    prepared_portfolio[
        "Product_Type"
    ]
    .isin(
        [
            "Mortgage",
            "Auto Loan",
        ]
    )
)

secured_account_count = int(
    secured_product_mask.sum()
)

missing_secured_collateral = int(
    (
        secured_product_mask
        &
        prepared_portfolio[
            "Collateral_Value"
        ].isna()
    )
    .sum()
)

# Audit flag only. Collateral is not imputed.
prepared_portfolio[
    "Secured_Collateral_Missing_Flag"
] = np.where(
    secured_product_mask
    &
    prepared_portfolio[
        "Collateral_Value"
    ].isna(),
    1,
    0
)

print("=" * 80)
print("STANDARDIZATION AND BUSINESS VALIDATION COMPLETED")
print("=" * 80)

print(
    "Invalid Customer Segments:",
    invalid_segments
)

print(
    "Invalid Product Types:",
    invalid_products
)

print(
    "Invalid Regions:",
    invalid_regions
)

print(
    "Invalid Reporting Dates:",
    invalid_reporting_dates
)

print(
    "Invalid Outstanding Balances:",
    invalid_balances
)

print(
    "Negative Remaining Terms:",
    invalid_remaining_terms
)

print(
    "Remaining Term Above Total Term:",
    remaining_above_total_term
)

print(
    "Secured Accounts:",
    secured_account_count
)

print(
    "Missing Secured Collateral:",
    missing_secured_collateral
)

print(
    "Collateral Imputation Performed: No"
)

display(
    prepared_portfolio[
        categorical_columns
    ]
    .drop_duplicates()
    .sort_values(
        categorical_columns
    )
)

logger.info(
    "Standardization and basic validation completed."
)

STANDARDIZATION AND BUSINESS VALIDATION COMPLETED
Invalid Customer Segments: 0
Invalid Product Types: 0
Invalid Regions: 0
Invalid Reporting Dates: 0
Invalid Outstanding Balances: 0
Negative Remaining Terms: 0
Remaining Term Above Total Term: 0
Secured Accounts: 4638
Missing Secured Collateral: 40
Collateral Imputation Performed: No


,Customer_Segment,Product_Type,Region
146,Corporate,Corporate Loan,Central
1,Corporate,Corporate Loan,East
401,Corporate,Corporate Loan,North
13,Corporate,Corporate Loan,South
140,Corporate,Corporate Loan,West
907,Corporate,Mortgage,Central
5700,Corporate,Mortgage,East
914,Corporate,Mortgage,North
314,Corporate,Mortgage,South
64,Corporate,Mortgage,West


2026-10-13 03:44:54,465 | INFO | Standardization and basic validation completed.


INFO:02_Data_Preparation:Standardization and basic validation completed.


In [7]:
# =============================================================================
# BLOCK 6: FINAL VALIDATION AND PREPARATION SUMMARY
# =============================================================================

output_record_count = len(
    prepared_portfolio
)

output_column_count = len(
    prepared_portfolio.columns
)

final_missing_credit_score = int(
    prepared_portfolio[
        "Credit_Score"
    ]
    .isna()
    .sum()
)

final_missing_current_dpd = int(
    prepared_portfolio[
        "Current_DPD"
    ]
    .isna()
    .sum()
)

duplicate_customer_ids = int(
    prepared_portfolio[
        "Customer_ID"
    ]
    .duplicated()
    .sum()
)

duplicate_loan_ids = int(
    prepared_portfolio[
        "Loan_ID"
    ]
    .duplicated()
    .sum()
)

validation_results = pd.DataFrame({
    "Validation_Check": [
        "Input and Output Record Count",
        "Missing Credit_Score After Preparation",
        "Missing Current_DPD After Preparation",
        "Duplicate Customer_ID",
        "Duplicate Loan_ID",
        "Invalid Reporting_Date",
    ],
    "Expected_Result": [
        input_record_count,
        0,
        0,
        0,
        0,
        0,
    ],
    "Actual_Result": [
        output_record_count,
        final_missing_credit_score,
        final_missing_current_dpd,
        duplicate_customer_ids,
        duplicate_loan_ids,
        invalid_reporting_dates,
    ],
})

validation_results[
    "Status"
] = np.where(
    validation_results[
        "Expected_Result"
    ]
    ==
    validation_results[
        "Actual_Result"
    ],
    "PASS",
    "FAIL"
)

processing_timestamp = datetime.now()

preparation_summary = pd.DataFrame({
    "Metric": [
        "Input Records",
        "Output Records",
        "Input Columns",
        "Output Columns",
        "Credit Scores Imputed",
        "Current DPD Values Imputed",
        "Credit Scores Corrected Below 300",
        "Credit Scores Corrected Above 900",
        "Negative DPD Values Corrected",
        "DPD Values Corrected Above 999",
        "Secured Accounts",
        "Missing Secured Collateral Preserved",
        "Processing Timestamp",
    ],
    "Result": [
        input_record_count,
        output_record_count,
        input_column_count,
        output_column_count,
        credit_scores_imputed,
        current_dpd_imputed,
        credit_score_below_300,
        credit_score_above_900,
        negative_current_dpd_before,
        current_dpd_above_999,
        secured_account_count,
        missing_secured_collateral,
        processing_timestamp,
    ],
})

print("=" * 80)
print("FINAL DATA PREPARATION VALIDATION")
print("=" * 80)

display(
    validation_results
)

print("\nData Preparation Summary:")

display(
    preparation_summary
)

failed_checks = validation_results.loc[
    validation_results[
        "Status"
    ] != "PASS"
]

if not failed_checks.empty:
    logger.error(
        "Module 02 final validation failed."
    )

    raise ValueError(
        "Module 02 validation failed. "
        "Review the failed checks."
    )

print(
    "Overall Module 02 Validation Status: PASS"
)

logger.info(
    "All Module 02 validation checks passed."
)

FINAL DATA PREPARATION VALIDATION


,Validation_Check,Expected_Result,Actual_Result,Status
0,Input and Output Record Count,10000,10000,PASS
1,Missing Credit_Score After Preparation,0,0,PASS
2,Missing Current_DPD After Preparation,0,0,PASS
3,Duplicate Customer_ID,0,0,PASS
4,Duplicate Loan_ID,0,0,PASS
5,Invalid Reporting_Date,0,0,PASS



Data Preparation Summary:


,Metric,Result
0,Input Records,10000
1,Output Records,10000
2,Input Columns,15
3,Output Columns,20
4,Credit Scores Imputed,100
5,Current DPD Values Imputed,75
6,Credit Scores Corrected Below 300,0
7,Credit Scores Corrected Above 900,0
8,Negative DPD Values Corrected,0
9,DPD Values Corrected Above 999,0


Overall Module 02 Validation Status: PASS
2026-10-13 03:47:24,572 | INFO | All Module 02 validation checks passed.


INFO:02_Data_Preparation:All Module 02 validation checks passed.


In [8]:
# =============================================================================
# BLOCK 7: SAVE OUTPUTS AND VERIFY FILES
# =============================================================================

try:
    # Save prepared portfolio.
    prepared_portfolio.to_csv(
        OUTPUT_FILE,
        index=False,
        date_format="%Y-%m-%d"
    )

    # Save preparation-level metrics.
    preparation_summary.to_csv(
        SUMMARY_FILE,
        index=False
    )

    # Save final validation evidence.
    validation_results.to_csv(
        VALIDATION_FILE,
        index=False
    )

except Exception as error:
    logger.exception(
        "Failed to save Module 02 outputs."
    )
    raise error

expected_files = [
    OUTPUT_FILE,
    SUMMARY_FILE,
    VALIDATION_FILE,
    LOG_FILE,
]

output_verification = pd.DataFrame({
    "File_Name": [
        file_path.name
        for file_path in expected_files
    ],
    "Exists": [
        file_path.exists()
        for file_path in expected_files
    ],
    "Size_Bytes": [
        (
            file_path.stat().st_size
            if file_path.exists()
            else 0
        )
        for file_path in expected_files
    ],
    "Full_Path": [
        str(file_path)
        for file_path in expected_files
    ],
})

logger.info(
    "Prepared portfolio saved successfully."
)

logger.info(
    "Module 02 completed successfully."
)

print("=" * 80)
print("PYTHON V2 - MODULE 02 COMPLETED")
print("DATA PREPARATION EXECUTED SUCCESSFULLY")
print("=" * 80)

print(
    "Input Records:",
    input_record_count
)

print(
    "Output Records:",
    output_record_count
)

print(
    "Credit Scores Imputed:",
    credit_scores_imputed
)

print(
    "Current DPD Values Imputed:",
    current_dpd_imputed
)

print(
    "Missing Secured Collateral Preserved:",
    missing_secured_collateral
)

print(
    "Prepared Output:",
    OUTPUT_FILE
)

print(
    "Preparation Summary:",
    SUMMARY_FILE
)

print(
    "Validation Evidence:",
    VALIDATION_FILE
)

print(
    "Execution Log:",
    LOG_FILE
)

print("=" * 80)

display(
    output_verification
)

assert output_verification[
    "Exists"
].all(), (
    "One or more Module 02 output files were not created."
)

print(
    "All Module 02 output files verified successfully."
)

2026-10-13 03:48:48,961 | INFO | Prepared portfolio saved successfully.


INFO:02_Data_Preparation:Prepared portfolio saved successfully.


2026-10-13 03:48:48,963 | INFO | Module 02 completed successfully.


INFO:02_Data_Preparation:Module 02 completed successfully.


PYTHON V2 - MODULE 02 COMPLETED
DATA PREPARATION EXECUTED SUCCESSFULLY
Input Records: 10000
Output Records: 10000
Credit Scores Imputed: 100
Current DPD Values Imputed: 75
Missing Secured Collateral Preserved: 40
Prepared Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/prepared_portfolio.csv
Preparation Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/02_data_preparation_summary.csv
Validation Evidence: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/02_data_preparation_validation.csv
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/02_Data_Preparation.log


,File_Name,Exists,Size_Bytes,Full_Path
0,prepared_portfolio.csv,True,1120457,/content/drive/MyDrive/Banking_Stress_Testing_...
1,02_data_preparation_summary.csv,True,393,/content/drive/MyDrive/Banking_Stress_Testing_...
2,02_data_preparation_validation.csv,True,286,/content/drive/MyDrive/Banking_Stress_Testing_...
3,02_Data_Preparation.log,True,581,/content/drive/MyDrive/Banking_Stress_Testing_...


All Module 02 output files verified successfully.
